### ***The chaos injection***

In this notebook the ***ideal*** baseline inventory is injected with data issues to make it akin to real data. 

The Chaos being injected to the data are:
- Temporal Chaos
- Schema and Structural Choas
- Operational and Logical Chaos
- Integrity and Market Chaos

In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

In [3]:
class SupplyChainChaosEngine:
    """
    Simulates specific systemic and operational failures in a Quick-Commerce supply chain dataset to test pipeline resilience.
    """
    def __init__(self,df_inventory, df_nodes):
        self.df = df_inventory.copy() # deep copy of baseline
        self.df_nodes = df_nodes.copy() # deep copy of hubs
        self.df['timestamp'] = pd.to_datetime(self.df['timestamp'])

    ### CATEGORY 1. TEMPORAL CHAOS (Time issues)
    def apply_temporal_chaos(self):
        # 1. Z-Time and IST trap: Shift 10% rows by 5.5 hours thus making it UTC (subtract 5 hours 30mins)
        idx1 = self.df.sample(frac=0.10).index
        self.df.loc[idx1,'timestamp'] -= timedelta(hours=5,minutes=30)

        # 2. Latency Smear: Massive lag on 2% of rows
        idx2 = self.df.sample(frac=0.02).index
        self.df.loc[idx2,'processing_lag_hours'] = 155.5
        return self
    
    ### CATEGORY 2: SCHEMA and STRUCTURAL CHAOS (Formatting issues)
    def apply_structural_chaos(self):
        # 3. Schema Poisoning: Inject "units" string into numberic "units_sold"
        idx = self.df.sample(n=50).index
        self.df.loc[idx,'units_sold'] = self.df.loc[idx,'units_sold'].astype(str) + " units"

         # 4. Address Emoji/Special Character corruption in nodes
        bad_chars = ["🌇", "~ ", "Phase 1\t", 'cityyy']
        for _ in range(3):
            self.df_nodes.loc[self.df_nodes.sample(1).index,'city'] = random.choice(bad_chars)

        # 5. Null value silent decay: drop reasons despite units_rto >0
        rto_mask = (self.df['units_rto'] > 0)
        null_idx = self.df[rto_mask].sample(n=20).index
        self.df.loc[null_idx, 'rto_reason'] = np.nan
        return self
    
    ### Category 3: Operational & Logical Chaos (Business Failures)
    def apply_logic_chaos(self):
        # 6. Ghost Inventory: Make units_sold > stock_on_hand
        ghost_idx = self.df.sample(n=20).index
        self.df.loc[ghost_idx, 'units_sold'] = self.df.loc[ghost_idx, 'stock_on_hand'] + 500

        # 7. Duplicate Transaction Echoes: Double-reporting 
        dupes = self.df.sample(n=100)
        self.df = pd.concat([self.df,dupes], ignore_index = True)

        # 8. Node Blackout: Delete 4 days of data for one specific node 
        self.df = self.df[~((self.df['node_id'] == 'NODE_006') & 
                            self.df['timestamp'].dt.day.isin([10,11,12,13]))] 
        
        # 9. Reverse Logistics Void: High rto but the stock doesn't reflect the return
        void_idx = self.df.sample(n=15).index
        self.df.loc[void_idx,'units_rto'] = 100
        self.df.loc[void_idx,'stock_on_hand'] = 2 # stock stays low
        return self
    
    ### Category 4: Integrity and Market chaos
    def apply_market_chaos(self):
        #10. SKU Migration / Identity Crisis
        mask = (self.df['product_id'] == 'PROD_001') & (self.df['timestamp'].dt.day < 10)
        self.df.loc[mask, 'product_id'] = 'PROD_001_OLD_SKU'

        # 11. Fat finger outliers : mistyping 
        self.df.loc[self.df.sample(n=2).index, 'units_sold'] = 999999
        return self
    
    def save_chaos(self):
        self.df.to_csv('03_inventory_chaotic.csv', index = False)
        self.df_nodes.to_csv('03_nodes_chaotic.csv', index = False)
        print("Success: Chaos injection completed.")


###### EXECUTION
# load baseline and nodes
df_inv = pd.read_csv('02_Test_Inventory_baseline.csv')
df_nod = pd.read_csv('01_Test_nodes.csv')


### RUN THE ENGINE
chaos = SupplyChainChaosEngine(df_inv, df_nod)
(chaos.apply_temporal_chaos().apply_structural_chaos().apply_logic_chaos().apply_market_chaos().save_chaos())

Success: Chaos injection completed.


C:\Users\loq\AppData\Local\Temp\ipykernel_6652\1967681361.py:25: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['9 units' '18 units' '31 units' '13 units' '22 units' '36 units'
 '8 units' '25 units' '24 units' '5 units' '10 units' '248 units'
 '272 units' '20 units' '12 units' '287 units' '19 units' '12 units'
 '13 units' '17 units' '10 units' '24 units' '286 units' '232 units'
 '20 units' '22 units' '23 units' '315 units' '28 units' '31 units'
 '29 units' '32 units' '40 units' '9 units' '22 units' '490 units'
 '9 units' '22 units' '20 units' '28 units' '11 units' '22 units'
 '137 units' '11 units' '27 units' '18 units' '365 units' '19 units'
 '117 units' '6 units']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  self.df.loc[idx,'units_sold'] = self.df.loc[idx,'units_sold'].astype(str) + " units"
